# StreamCAG Demo Notebook
Complete implementation of StreamCAG with visualizations

In [ ]:
import sys
sys.path.append('..')

from streamcag import StreamCAG, StreamCAGConfig
from streamcag.utils import create_benchmark_dataset, analyze_results
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time

# Configuration
config = StreamCAGConfig(
    model_name='mistralai/Mistral-7B-Instruct-v0.1',
    cache_max_size=500,
    use_l2_cache=True,
    cache_storage_path='./streamcag_demo_cache',
    optimize_context=True,
    target_context_tokens=1024,
    use_4bit_quantization=True
)

# Initialize StreamCAG
print('Initializing StreamCAG...')
cag = StreamCAG(config)
print('StreamCAG initialized successfully.')

# Preload knowledge
print('Preloading knowledge into cache...')
knowledge_docs = [
    'Paris is the capital and most populous city of France. It is located in the north-central part of the country along the Seine River. Paris is known for its art, fashion, gastronomy, and culture.',
    'Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Unlike traditional programming where rules are explicitly coded, machine learning algorithms identify patterns in data and make predictions or decisions.',
    'The water cycle, also known as the hydrological cycle, describes the continuous movement of water on, above, and below the surface of the Earth. The main processes are evaporation, condensation, precipitation, and runoff.',
    'Blockchain is a distributed ledger technology that maintains a secure and decentralized record of transactions. It is the underlying technology behind cryptocurrencies like Bitcoin and enables secure peer-to-peer transactions without intermediaries.',
    'Quantum computing uses quantum bits or qubits, which can exist in multiple states simultaneously (superposition). This allows quantum computers to solve certain problems much faster than classical computers, particularly in cryptography and optimization.'
]

metadata = [
    {'source': 'geography', 'topic': 'cities'},
    {'source': 'technology', 'topic': 'AI'},
    {'source': 'science', 'topic': 'earth science'},
    {'source': 'technology', 'topic': 'cryptocurrency'},
    {'source': 'technology', 'topic': 'quantum physics'}
]

cag.preload_knowledge(knowledge_docs, metadata)
print('Knowledge preloaded.')

# Interactive query loop
print('StreamCAG is ready. Running test queries...')

questions = [
    'What is the capital of France?',
    'How does machine learning differ from traditional programming?',
    'Explain the water cycle briefly.',
    'What is blockchain technology?',
    'What are qubits in quantum computing?'
]

results = []
for i, question in enumerate(questions, 1):
    print('=' * 60)
    print(f'Query {i}: {question}')
    print('=' * 60)
    
    # With cache
    print('With cache augmentation:')
    result_with_cache = cag.query(question, use_cache=True)
    print(f"Answer: {result_with_cache['answer']}")
    print(f"Context used: {result_with_cache['context_used']}")
    print(f"Response time: {result_with_cache['response_time']:.2f}s")
    
    # Without cache (for comparison)
    print('Without cache (for comparison):')
    result_without_cache = cag.query(question, use_cache=False)
    print(f"Answer: {result_without_cache['answer'][:200]}...")
    
    results.append({
        'question': question,
        'with_cache': result_with_cache,
        'without_cache': result_without_cache
    })
    
    time.sleep(1)

# Benchmarking
print('Running benchmark...')
benchmark_dataset = create_benchmark_dataset(20)
benchmark_results = []

for item in benchmark_dataset:
    result = cag.query(item['question'])
    benchmark_results.append({
        'question': item['question'],
        'response_time': result['response_time'],
        'context_used': result['context_used'],
        'cache_hits': result['cache_hits'],
        'optimization_stats': result.get('optimization_stats')
    })

# Analysis
print('Performance Analysis:')
stats_df = analyze_results(benchmark_results)
print(stats_df)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Response time distribution
axes[0, 0].hist([r['response_time'] for r in benchmark_results], bins=10, edgecolor='black')
axes[0, 0].set_title('Response Time Distribution')
axes[0, 0].set_xlabel('Time (seconds)')
axes[0, 0].set_ylabel('Frequency')

# Cache hit rate
cache_hits = [r['context_used'] for r in benchmark_results]
axes[0, 1].pie([sum(cache_hits), len(cache_hits)-sum(cache_hits)],
               labels=['Cache Hit', 'Cache Miss'],
               autopct='%1.1f%%',
               colors=['#4CAF50', '#F44336'])
axes[0, 1].set_title('Cache Hit Rate')

# Token savings
if all(r.get('optimization_stats') for r in benchmark_results):
    before = [r['optimization_stats']['context_tokens_before'] for r in benchmark_results if r.get('optimization_stats')]
    after = [r['optimization_stats']['context_tokens_after'] for r in benchmark_results if r.get('optimization_stats')]
    
    axes[1, 0].bar(['Before', 'After'], [sum(before)/len(before), sum(after)/len(after)])
    axes[1, 0].set_title('Average Context Tokens')
    axes[1, 0].set_ylabel('Tokens')

# System stats
system_stats = cag.get_system_stats()
stats_items = ['cache_hit_rate', 'token_savings_percentage', 'avg_response_time']
stats_values = [system_stats.get(k, 0) for k in stats_items]
stats_labels = ['Hit Rate (%)', 'Token Savings (%)', 'Avg Time (s)']

axes[1, 1].bar(stats_labels, stats_values)
axes[1, 1].set_title('System Performance Metrics')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Display comprehensive statistics
print('StreamCAG System Statistics:')
print(json.dumps(cag.get_system_stats(), indent=2))

# Save caches
cag.save_caches()
print('Caches saved to disk.')

print('Demo completed successfully.')
